# Enterprise Migration Copilot — Benchmark

Evaluates 6 models (3 base + 3 fine-tuned) on 480 held-out test scripts.
Runs models locally in Colab — no external API needed.

**Setup:** Runtime → T4 GPU · Add `HF_TOKEN` to Colab Secrets (🔑 sidebar)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.46.0 peft==0.13.0 accelerate==0.34.0 datasets huggingface_hub
!pip install -q numpy==1.26.4

import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 2 — Configuration
HF_USERNAME = "praveends"

MODELS = {
    "DeepSeek-1.3B-base": {
        "repo": "deepseek-ai/deepseek-coder-1.3b-instruct",
        "adapter": None,
        "is_finetuned": False,
        "family": "deepseek",
    },
    "DeepSeek-1.3B-finetuned": {
        "repo": "deepseek-ai/deepseek-coder-1.3b-instruct",
        "adapter": f"{HF_USERNAME}/migration-copilot-deepseek-coder-1-3b-instruct",
        "is_finetuned": True,
        "family": "deepseek",
    },
    "Qwen2.5-1.5B-base": {
        "repo": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "adapter": None,
        "is_finetuned": False,
        "family": "qwen",
    },
    "Qwen2.5-1.5B-finetuned": {
        "repo": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "adapter": f"{HF_USERNAME}/migration-copilot-qwen2-5-coder-1-5b-instruct",
        "is_finetuned": True,
        "family": "qwen",
    },
    "Phi-3.5-mini-base": {
        "repo": "microsoft/Phi-3.5-mini-instruct",
        "adapter": None,
        "is_finetuned": False,
        "family": "phi",
    },
    "Phi-3.5-mini-finetuned": {
        "repo": "microsoft/Phi-3.5-mini-instruct",
        "adapter": f"{HF_USERNAME}/migration-copilot-phi-3-5-mini-instruct",
        "is_finetuned": True,
        "family": "phi",
    },
}

print("Models configured:")
for name, cfg in MODELS.items():
    kind = "fine-tuned" if cfg["is_finetuned"] else "base"
    print(f"  {name} ({kind})")

In [ ]:
# Cell 3 — HuggingFace login
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to HuggingFace")

In [ ]:
# Cell 4 — Load test split from HuggingFace dataset
import json
from datasets import load_dataset
from collections import defaultdict
import random

HF_USERNAME = "praveends"

print("Loading dataset from HuggingFace...")
dataset = load_dataset(f"{HF_USERNAME}/enterprise-migration-dataset",
                       data_files="train.jsonl", split="train")
print(f"Total pairs: {len(dataset)}")

# Create stratified test split (same logic as create_test_split.py)
TARGET_COUNTS = {
    "sql":              {"easy": 30, "medium": 40, "hard": 35, "expert": 15},
    "hiveql":           {"easy": 30, "medium": 40, "hard": 35, "expert": 15},
    "plsql":            {"easy": 30, "medium": 40, "hard": 35, "expert": 15},
    "stored_procedure": {"easy": 40, "medium": 40, "hard": 30, "expert": 10},
}

random.seed(99)
buckets = defaultdict(lambda: defaultdict(list))
for pair in dataset:
    lang = pair.get("source_language", "")
    diff = pair.get("difficulty", "")
    if lang in TARGET_COUNTS and diff in TARGET_COUNTS.get(lang, {}):
        buckets[lang][diff].append(dict(pair))

test_pairs = []
for lang, diff_counts in TARGET_COUNTS.items():
    for diff, count in diff_counts.items():
        pool = buckets[lang][diff]
        sampled = random.sample(pool, min(count, len(pool)))
        test_pairs.extend(sampled)

random.shuffle(test_pairs)
print(f"Test split: {len(test_pairs)} scripts")

by_lang = defaultdict(int)
for p in test_pairs:
    by_lang[p['source_language']] += 1
print("By language:", dict(by_lang))

In [ ]:
# Cell 5 — Scoring functions
import ast
import re

DATAFRAME_OPS = [
    r"\.select\(", r"\.filter\(", r"\.where\(", r"\.groupBy\(",
    r"\.join\(", r"\.agg\(", r"\.withColumn\(", r"\.orderBy\(",
    r"\.union\(", r"\.distinct\(", r"\.drop\(", r"\.limit\(",
    r"spark\.table\(", r"spark\.read", r"spark\.sql\(",
    r"\.show\(", r"\.count\(", r"\.collect\(", r"\.write\.",
    r"\.repartition\(", r"\.cache\(", r"DeltaTable\.",
]

def build_prompt(example):
    source_lang = example.get('source_language', 'sql').upper()
    difficulty = example.get('difficulty', 'medium')
    source_code = example.get('source_code', '')
    return f"""### Instruction:
Convert the following {source_lang} code to PySpark.
Difficulty: {difficulty}

### Input:
{source_code}

### Response:
"""

def score_output(generated, example):
    # Metric 1: Syntax valid
    syntax_valid = False
    try:
        ast.parse(generated)
        syntax_valid = True
    except SyntaxError:
        pass

    # Metric 2: At least 2 DataFrame ops
    op_count = sum(1 for p in DATAFRAME_OPS if re.search(p, generated))
    has_pyspark_ops = op_count >= 2

    # Metric 3: Table name alignment
    source_code = example.get('source_code', '').lower()
    table_pattern = r'(?:from|join)\s+([a-z_][a-z0-9_]*)'
    source_tables = set(re.findall(table_pattern, source_code))
    if source_tables:
        found = sum(1 for t in source_tables if t in generated.lower())
        semantic_sim = (found / len(source_tables)) >= 0.60
    else:
        semantic_sim = True

    return {
        "syntax_valid": syntax_valid,
        "has_pyspark_ops": has_pyspark_ops,
        "semantic_sim": semantic_sim,
        "overall": syntax_valid and has_pyspark_ops and semantic_sim,
        "op_count": op_count,
    }

print("Scoring functions ready")

In [ ]:
# Cell 6 — Run benchmark (loads each model, runs all test scripts, unloads)
import torch
import gc
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from collections import defaultdict

all_results = []

# Set DRY_RUN = True to test with 12 scripts before the full 480-script run
DRY_RUN = False
scripts = test_pairs[:12] if DRY_RUN else test_pairs
print(f"Running benchmark on {len(scripts)} scripts across {len(MODELS)} models")
print(f"Total inference calls: {len(scripts) * len(MODELS)}")

for model_name, config in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"Base: {config['repo']}")
    if config['adapter']:
        print(f"Adapter: {config['adapter']}")
    print(f"{'='*60}")

    # Load tokenizer + base model
    tokenizer = AutoTokenizer.from_pretrained(
        config['repo'], trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        config['repo'],
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    # Load LoRA adapter if fine-tuned
    if config['adapter']:
        model = PeftModel.from_pretrained(model, config['adapter'])
        model = model.merge_and_unload()  # merge for faster inference
        print("LoRA adapter merged")

    model.eval()
    print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    # Run inference on all test scripts
    model_results = []
    for i, example in enumerate(scripts):
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt",
                          truncation=True, max_length=512).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = response[len(prompt):].strip()

        scores = score_output(generated, example)
        scores.update({
            "model": model_name,
            "is_finetuned": config["is_finetuned"],
            "family": config["family"],
            "source_language": example.get("source_language", ""),
            "difficulty": example.get("difficulty", ""),
        })
        model_results.append(scores)
        all_results.append(scores)

        if (i + 1) % 50 == 0:
            passed = sum(1 for r in model_results if r['overall'])
            print(f"  [{i+1}/{len(scripts)}] pass rate so far: {passed/(i+1):.1%}")

    # Summary for this model
    passed = sum(1 for r in model_results if r['overall'])
    print(f"  FINAL: {passed}/{len(model_results)} passed ({passed/len(model_results):.1%})")

    # Unload model to free VRAM before next model
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Model unloaded. VRAM freed.")

print(f"\nBenchmark complete. Total results: {len(all_results)}")

In [ ]:
# Cell 7 — Save results and write leaderboard
import json
from datetime import datetime
from google.colab import files

RESULTS_PATH = "/content/results.json"
SUMMARY_PATH = "/content/summary.md"

# Save raw results
with open(RESULTS_PATH, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved: {RESULTS_PATH}")

# Build leaderboard
languages = ["sql", "hiveql", "plsql", "stored_procedure"]
model_names = list(MODELS.keys())

def get_rate(model, lang=None):
    subset = [r for r in all_results if r["model"] == model]
    if lang:
        subset = [r for r in subset if r["source_language"] == lang]
    return sum(1 for r in subset if r["overall"]) / len(subset) if subset else 0

# Find best fine-tuned model
finetuned = [m for m in model_names if MODELS[m]["is_finetuned"]]
best_model = max(finetuned, key=lambda m: get_rate(m))
best_rate = get_rate(best_model)

print(f"\nBest fine-tuned model: {best_model} ({best_rate:.1%})")

# Print leaderboard to console
print(f"\n{'Model':35s} {'SQL':6s} {'HiveQL':8s} {'PL/SQL':8s} {'SP':6s} {'Overall':8s}")
print("-" * 75)
for m in model_names:
    row = f"{m:35s}"
    for lang in languages:
        row += f" {get_rate(m, lang):.0%}    "
    row += f" {get_rate(m):.0%}"
    marker = " ⭐" if m == best_model else ""
    print(row + marker)

# Write summary.md
summary = f"""# Benchmark Results — Enterprise Migration Copilot

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Test set: {len(scripts)} scripts | Metrics: syntax_valid AND has_pyspark_ops AND semantic_sim

## Leaderboard

| Model | SQL | HiveQL | PL/SQL | SP | Overall |
|-------|-----|--------|--------|----|---------|
"""
for m in model_names:
    marker = " ⭐" if m == best_model else ""
    row_vals = " | ".join(f"{get_rate(m, lang):.0%}" for lang in languages)
    summary += f"| {m+marker} | {row_vals} | {get_rate(m):.0%} |\n"

summary += f"""
## Fine-tuning Improvement

| Family | Base | Fine-tuned | Delta |
|--------|------|------------|-------|
"""
for family in ["deepseek", "qwen", "phi"]:
    base = next(m for m in model_names if MODELS[m]["family"] == family and not MODELS[m]["is_finetuned"])
    ft = next(m for m in model_names if MODELS[m]["family"] == family and MODELS[m]["is_finetuned"])
    delta = get_rate(ft) - get_rate(base)
    sign = "+" if delta >= 0 else ""
    summary += f"| {family.capitalize()} | {get_rate(base):.1%} | {get_rate(ft):.1%} | {sign}{delta:.1%} |\n"

summary += f"\n## Best Model\n\n**{best_model}** — {best_rate:.1%} overall pass rate\n"

with open(SUMMARY_PATH, "w") as f:
    f.write(summary)
print(f"\nSummary written: {SUMMARY_PATH}")

# Auto-download both files
files.download(RESULTS_PATH)
files.download(SUMMARY_PATH)
print("Files downloading to your machine...")

In [ ]:
# Cell 8 — Failure analysis for best fine-tuned model
from collections import defaultdict
from google.colab import files

FAILURE_PATH = "/content/failure_analysis.md"

best_results = [r for r in all_results if r["model"] == best_model]
failures = [r for r in best_results if not r["overall"]]
passes = [r for r in best_results if r["overall"]]

def categorize(result):
    cats = []
    if not result.get("syntax_valid"): cats.append("syntax_error")
    if not result.get("has_pyspark_ops"): cats.append("missing_dataframe_ops")
    if not result.get("semantic_sim"): cats.append("low_semantic_alignment")
    return cats or ["unknown"]

failure_cats = defaultdict(lambda: defaultdict(int))
for f in failures:
    lang = f.get("source_language", "unknown")
    for cat in categorize(f):
        failure_cats[lang][cat] += 1

print(f"Best model: {best_model}")
print(f"Pass rate: {len(passes)}/{len(best_results)} ({len(passes)/len(best_results):.1%})")
print()

report = f"""# Failure Analysis — {best_model}

| Metric | Value |
|--------|-------|
| Model | {best_model} |
| Total scripts | {len(best_results)} |
| Passed | {len(passes)} ({len(passes)/len(best_results):.1%}) |
| Failed | {len(failures)} ({len(failures)/len(best_results):.1%}) |

## Per-Language Results

| Language | Total | Passed | Failed | Pass Rate |
|----------|-------|--------|--------|-----------|
"""

for lang in ["sql", "hiveql", "plsql", "stored_procedure"]:
    lang_r = [r for r in best_results if r["source_language"] == lang]
    lang_p = sum(1 for r in lang_r if r["overall"])
    lang_f = len(lang_r) - lang_p
    rate = lang_p / len(lang_r) if lang_r else 0
    report += f"| {lang:20s} | {len(lang_r):5d} | {lang_p:6d} | {lang_f:6d} | {rate:.1%} |\n"
    print(f"{lang:20s}: {lang_p}/{len(lang_r)} ({rate:.1%})")

report += "\n## Failure Categories\n\n"
for lang in ["sql", "hiveql", "plsql", "stored_procedure"]:
    cats = failure_cats.get(lang, {})
    if cats:
        report += f"### {lang.upper()}\n"
        for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
            report += f"- `{cat}`: {count} failures\n"
        report += "\n"

with open(FAILURE_PATH, "w") as f:
    f.write(report)
print(f"\nFailure analysis saved: {FAILURE_PATH}")

# Auto-download
files.download(FAILURE_PATH)
print("Failure analysis downloading...")

## After Running

Results saved to Google Drive at `migration-benchmark/`:
- `results.json` — all individual scores
- `summary.md` — leaderboard table
- `failure_analysis.md` — best model failure breakdown

Download these files and copy to your local repo:
- `results.json` → `benchmark/results.json`
- `summary.md` → `benchmark/summary.md`
- `failure_analysis.md` → `benchmark/failure_analysis.md`